<a href="https://colab.research.google.com/github/jovanfranco/usdmxn-volatilidad/blob/main/clase22_largo_corto_vol_objetivo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Clase 22: posiciones largas y cortas con volatilidad objetivo (FX)

Pesos proporcionales a inversa(cov) por mu, escalados a una volatilidad objetivo. Sin carry todavia.

In [2]:
import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.covariance import LedoitWolf

# Ejemplo a mano: dos activos, posiciones largas y cortas con volatilidad objetivo
mu = np.array([0.04, 0.02])          # rendimientos esperados anuales (en exceso: posiciones que no cuestan)
vol = np.array([0.10, 0.20])
rho = 0.3
cov = np.outer(vol, vol) * np.array([[1, rho], [rho, 1]])
print("Covarianza:\n", cov)

x = np.linalg.solve(cov, mu)         # direccion: inversa(cov) por mu
print("Direccion x (sin escala):", x.round(3))
vol_x = float(np.sqrt(x @ cov @ x))
print("Volatilidad de x:", round(vol_x, 4))
vol_obj = 0.08
w = x * (vol_obj / vol_x)            # escalamos para llegar a la volatilidad objetivo
print("Pesos con volatilidad objetivo de 8%:", w.round(3), "| suma:", round(w.sum(), 3), "| exposicion bruta:", round(np.abs(w).sum(), 3))
print("Volatilidad de la cartera:", round(float(np.sqrt(w @ cov @ w)), 4), "| rendimiento esperado:", round(float(w @ mu), 4),
      "| Sharpe:", round(float(w @ mu / np.sqrt(w @ cov @ w)), 3))

# El Sharpe no depende de la volatilidad objetivo: solo cambia la escala
for objetivo in (0.04, 0.08, 0.16):
    wk = x * (objetivo / vol_x)
    print("Objetivo", objetivo, "-> rendimiento", round(float(wk @ mu), 4), "| Sharpe", round(float(wk @ mu / np.sqrt(wk @ cov @ wk)), 3))

# Ahora un caso donde el segundo activo tiene rendimiento esperado MENOR que lo que "le toca" por su correlacion: se vuelve corto
mu2 = np.array([0.04, 0.005])
x2 = np.linalg.solve(cov, mu2)
w2 = x2 * (vol_obj / np.sqrt(x2 @ cov @ x2))
print("\nCon rendimientos esperados", mu2, "-> pesos:", w2.round(3), "(el segundo queda corto)")


Covarianza:
 [[0.01  0.006]
 [0.006 0.04 ]]
Direccion x (sin escala): [ 4.066 -0.11 ]
Volatilidad de x: 0.4005
Pesos con volatilidad objetivo de 8%: [ 0.812 -0.022] | suma: 0.79 | exposicion bruta: 0.834
Volatilidad de la cartera: 0.08 | rendimiento esperado: 0.032 | Sharpe: 0.401
Objetivo 0.04 -> rendimiento 0.016 | Sharpe 0.401
Objetivo 0.08 -> rendimiento 0.032 | Sharpe 0.401
Objetivo 0.16 -> rendimiento 0.0641 | Sharpe 0.401

Con rendimientos esperados [0.04  0.005] -> pesos: [ 0.837 -0.101] (el segundo queda corto)


In [3]:
# Funciones
def pesos_ls(mu, cov, vol_obj=0.08, bruto_max=None):
    """Largo/corto: pesos proporcionales a inversa(cov) por mu, escalados a la volatilidad objetivo.
    Si bruto_max se da y la exposicion bruta (suma de valores absolutos) lo rebasa, se reduce (la volatilidad queda por debajo)."""
    x = np.linalg.solve(cov, mu)
    w = x * (vol_obj / np.sqrt(x @ cov @ x))
    if bruto_max is not None and np.abs(w).sum() > bruto_max:
        w = w * (bruto_max / np.abs(w).sum())
    return w

def mu_desde_sharpe(sharpe, vols, lam):
    """Encoge los Sharpe individuales hacia el Sharpe promedio y los convierte en rendimientos: mu = Sharpe * vol."""
    s = lam * sharpe + (1 - lam) * sharpe.mean()
    return s * vols

# El encogimiento hacia CERO no sirve con volatilidad objetivo: la escala se cancela
w_a = pesos_ls(mu, cov)
w_b = pesos_ls(0.25 * mu, cov)
print("Pesos con mu:", w_a.round(4), "| con 0.25 * mu:", w_b.round(4), "-> iguales:", np.allclose(w_a, w_b))


Pesos con mu: [ 0.8121 -0.0219] | con 0.25 * mu: [ 0.8121 -0.0219] -> iguales: True


In [4]:
# Datos reales: seis pares contra el dolar (precio solo, sin carry todavia)
pares = {"USD/MXN": ("MXN=X", False), "USD/JPY": ("JPY=X", False), "USD/EUR": ("EURUSD=X", True),
         "USD/GBP": ("GBPUSD=X", True), "USD/CAD": ("CAD=X", False), "USD/BRL": ("BRL=X", False)}
precios = {}
for nombre, (ticker, invertir) in pares.items():
    p = yf.download(ticker, start="2020-10-07", end="2026-10-22", progress=False)["Close"].squeeze()
    precios[nombre] = 1 / p if invertir else p
rend = pd.DataFrame(precios).dropna().pct_change().dropna()
corte = int(len(rend) * 0.7)
entren, prueba = rend.iloc[:corte], rend.iloc[corte:]
print("Dias:", len(rend), "| del", rend.index[0].date(), "al", rend.index[-1].date(),
      "| entrenamiento:", len(entren), "| prueba:", len(prueba))

mu_e = entren.mean().values * 252
vols_e = entren.std().values * np.sqrt(252)
cov_e = LedoitWolf().fit(entren.values).covariance_ * 252
sharpe_e = mu_e / vols_e
print("Sharpe individual en entrenamiento:")
print(pd.Series(sharpe_e, index=rend.columns).round(2).to_string())

VOL_OBJ = 0.08
n = len(rend.columns)
carteras = {"Largo USD, pesos iguales": np.ones(n) / n}
carteras["Largo USD, pesos iguales"] = carteras["Largo USD, pesos iguales"] * VOL_OBJ / np.sqrt(carteras["Largo USD, pesos iguales"] @ cov_e @ carteras["Largo USD, pesos iguales"])
carteras["L/C con mu sin encoger"] = pesos_ls(mu_e, cov_e, VOL_OBJ)
carteras["L/C con Sharpe encogido (0.5)"] = pesos_ls(mu_desde_sharpe(sharpe_e, vols_e, 0.5), cov_e, VOL_OBJ)
carteras["L/C con Sharpe igual (0)"] = pesos_ls(mu_desde_sharpe(sharpe_e, vols_e, 0.0), cov_e, VOL_OBJ)
pesos = pd.DataFrame(carteras, index=rend.columns)
print("\nPesos (volatilidad objetivo de 8%, estimada con el entrenamiento):")
print(pesos.round(3).to_string())
print("Exposicion bruta:", pesos.abs().sum().round(2).to_dict())

filas = []
for nombre, w in carteras.items():
    s_in, s_out = entren.values @ w, prueba.values @ w
    filas.append({"cartera": nombre,
                  "vol_dentro_%": s_in.std() * np.sqrt(252) * 100,
                  "vol_fuera_%": s_out.std() * np.sqrt(252) * 100,
                  "Sharpe_dentro": s_in.mean() / s_in.std() * np.sqrt(252),
                  "rend_fuera_%": s_out.mean() * 252 * 100,
                  "Sharpe_fuera": s_out.mean() / s_out.std() * np.sqrt(252)})
print("\nDesempeno (objetivo de volatilidad: 8%):")
print(pd.DataFrame(filas).round(2).to_string(index=False))
anios = len(prueba) / 252
print("Error estandar aproximado del Sharpe fuera de muestra:", round(1 / np.sqrt(anios), 2))


/tmp/ipykernel_418/3169467221.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  p = yf.download(ticker, start="2020-10-07", end="2026-10-22", progress=False)["Close"].squeeze()
/tmp/ipykernel_418/3169467221.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  p = yf.download(ticker, start="2020-10-07", end="2026-10-22", progress=False)["Close"].squeeze()
/tmp/ipykernel_418/3169467221.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  p = yf.download(ticker, start="2020-10-07", end="2026-10-22", progress=False)["Close"].squeeze()
/tmp/ipykernel_418/3169467221.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  p = yf.download(ticker, start="2020-10-07", end="2026-10-22", progress=False)["Close"].squeeze()
/tmp/ipykernel_418/3169467221.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  p = yf.download(ticker, start="2

Dias: 1570 | del 2020-10-08 al 2026-10-21 | entrenamiento: 1099 | prueba: 471
Sharpe individual en entrenamiento:
USD/MXN   -0.09
USD/JPY    1.02
USD/EUR    0.41
USD/GBP    0.12
USD/CAD    0.29
USD/BRL    0.23

Pesos (volatilidad objetivo de 8%, estimada con el entrenamiento):
         Largo USD, pesos iguales  L/C con mu sin encoger  L/C con Sharpe encogido (0.5)  L/C con Sharpe igual (0)
USD/MXN                     0.201                  -0.173                         -0.056                     0.237
USD/JPY                     0.201                   0.810                          0.763                     0.397
USD/EUR                     0.201                   0.358                          0.315                     0.111
USD/GBP                     0.201                  -0.575                         -0.432                     0.046
USD/CAD                     0.201                   0.270                          0.290                     0.238
USD/BRL                     0.20

In [5]:
# Volatilidad objetivo: ¿que tanto se cumple?  Volatilidad realizada en ventanas de 63 dias del periodo de prueba
for nombre, w in carteras.items():
    serie = pd.Series(prueba.values @ w, index=prueba.index)
    v = serie.rolling(63).std().dropna() * np.sqrt(252) * 100
    print(f"{nombre}: vol realizada 63d  min {v.min():.1f} | mediana {v.median():.1f} | max {v.max():.1f}  (objetivo 8.0)")


Largo USD, pesos iguales: vol realizada 63d  min 4.3 | mediana 6.9 | max 10.3  (objetivo 8.0)
L/C con mu sin encoger: vol realizada 63d  min 3.0 | mediana 7.3 | max 11.6  (objetivo 8.0)
L/C con Sharpe encogido (0.5): vol realizada 63d  min 3.3 | mediana 7.6 | max 11.2  (objetivo 8.0)
L/C con Sharpe igual (0): vol realizada 63d  min 4.5 | mediana 7.1 | max 10.2  (objetivo 8.0)


In [6]:
# Carry: opcional. Cuando tengas las tasas historicas, llena este DataFrame y vuelve a correr.
# Debe tener las mismas fechas que `rend` (o mas) y una columna por par con el carry DIARIO en decimal:
# carry diario del par USD/X = (tasa USD - tasa X) / 100 / 252, con las tasas de ESA fecha (no las de hoy).
carry_diario = None   # ejemplo: pd.DataFrame(..., index=fechas, columns=rend.columns)

if carry_diario is None:
    print("Sin carry: los rendimientos de arriba son solo de precio. No se publica nada de carry todavia.")
else:
    rend_total = rend + carry_diario.reindex(rend.index).ffill()
    print("Carry anual promedio por par (%):")
    print((carry_diario.reindex(rend.index).ffill().mean() * 252 * 100).round(2).to_string())
    # Aqui repetirias CELDA 3 usando rend_total en lugar de rend.


Sin carry: los rendimientos de arriba son solo de precio. No se publica nada de carry todavia.
